# Dataset Validation

Notebook configurabile per verificare la distribuzione delle classi nei dataset di segmentazione chirurgica.

Il notebook calcola **sull'intero dataset**, senza sampling, a tre livelli:
- **per split** (train / validation / test);
- **per video** (dentro ogni split);
- **globale** (tutti i campioni del dataset).

Per ogni livello:
- numero di esempi, numero e percentuale di esempi in cui compare ogni classe;
- numero totale di pixel per classe e percentuale sui pixel validi;
- `ignored_pixels`: pixel con etichetta fuori dall'intervallo delle classi (non entrano nelle percentuali).

La logica di calcolo è in `src/datasets/validation.py` (ogni maschera viene letta una sola volta). Se il dataset possiede split ufficiali/metadati vengono usati quelli; altrimenti viene usata la configurazione Hydra.

# CONFIGURAZIONE PRINCIPALE

In [1]:
# Change ONLY this
#   "baseline/cholec_seg8k"
#   "baseline/dsad"
#   "baseline/sarrarp50"
#   etc

DATASET_CONFIG = "baseline/cholec_seg8k"

# Nome del config principale Hydra del progetto.
HYDRA_CONFIG_NAME = "config"

PROJECT_ROOT = None

# Se True, stampa informazioni utili per il debug.
DEBUG = True


In [2]:
# ============================================================
# SETUP PROGETTO
# ============================================================

from pathlib import Path
import sys


def find_project_root(start=None):
    """Find the repository root containing configs/ and src/."""
    start = Path(start or Path.cwd()).resolve()

    candidates = [start, *start.parents]

    for path in candidates:
        if (path / "src").is_dir() and (path / "configs").is_dir():
            return path

    raise FileNotFoundError(
        "Impossibile trovare la root della repo. "
        "Imposta PROJECT_ROOT manualmente nella cella precedente."
    )


if PROJECT_ROOT is None:
    PROJECT_ROOT = find_project_root()
else:
    PROJECT_ROOT = Path(PROJECT_ROOT).resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset config: {DATASET_CONFIG}")
print(f"Hydra config: {HYDRA_CONFIG_NAME}")


Project root: /homes/cmininno/tesi/surgical_scene_segmentation
Dataset config: baseline/cholec_seg8k
Hydra config: config


In [3]:
# ============================================================
# IMPORT
# ============================================================

import numpy as np
import pandas as pd
from omegaconf import OmegaConf
                
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate


In [ ]:
# ============================================================
# CARICAMENTO CONFIG + DATASET
# ============================================================

config_dir = PROJECT_ROOT / "configs"

with initialize_config_dir(
    version_base=None,
    config_dir=str(config_dir),
):
    cfg = compose(
        config_name=HYDRA_CONFIG_NAME,
        overrides=[f"dataset={DATASET_CONFIG}"],
    )

print("Configurazione caricata:")
print(OmegaConf.to_yaml(cfg))


dataset = instantiate(cfg.data.dataset)

print(f"Dataset: {type(dataset).__name__}")
print(f"Samples: {len(dataset.samples)}")
print(f"Classes: {dataset.class_names}")


Configurazione caricata:
name: cholecseg8k
data:
  name: cholecseg8k
  dataset:
    _target_: src.datasets.cholecseg8k.CholecSeg8kDataset
    root: ${paths.data}/CholecSeg8k
  num_classes: 13
  split:
    strategy: percentage
    group_by: video
    seed: 42
    stratified: true
    train_ratio: 0.7
    val_ratio: 0.3
    test_ratio: 0.0
preprocessing:
  image:
    size:
    - 240
    - 427
    channel_first: true
    dtype: float32
    normalize: true
  mask:
    channel_first: true
    dtype: long
  padding:
    enabled: true
    divisor: 16
augmentation:
  train:
    enabled: true
    horizontal_flip:
      enabled: true
      probability: 0.5
    scaling:
      enabled: true
      range:
      - -0.1
      - 0.1
      probability: 1.0
    gaussian_blur:
      enabled: true
      probability: 1.0
      kernel_sizes:
      - 3
      - 5
      - 7
      sigmas:
      - 0.8
      - 1.1
      - 1.4
    random_crop:
      enabled: true
      size:
      - 256
      - 256
  validation:
  

In [ ]:
# ============================================================
# FUNZIONI DI SUPPORTO (dal modulo condiviso)
# ============================================================

from src.datasets.validation import (
    get_samples,
    get_class_names,
    _to_numpy as to_numpy,
    _load_raw_sample as load_raw_sample,
    resolve_splits,
    compute_sample_counts,
    class_pixel_statistics,
    class_statistics_by_video,
    class_statistics_global,
    class_video_presence,
    class_video_matrix,
)


## Risoluzione degli split

La regola è semplice:

1. se `dataset.samples` contiene `split`, vengono usati gli split già definiti dal dataset;
2. altrimenti vengono usati gli split generati dalla configurazione Hydra.

Non viene applicato alcun limite al numero di campioni.

In [ ]:
# ============================================================
# RISOLUZIONE SPLIT
# ============================================================

splits = resolve_splits(dataset, cfg)

split_summary = pd.DataFrame([
    {"split": split_name, "samples": len(indices)}
    for split_name, indices in splits.items()
])

display(split_summary)


## Distribuzione delle classi

**Vengono analizzati tutti gli esempi di ogni split.** Ogni maschera viene letta una sola volta e i conteggi sono riutilizzati per le tabelle per split, per video e globale.

- `samples` = numero di immagini/mask in cui compare la classe.
- `sample_percentage` = percentuale delle immagini in cui compare la classe.
- `pixels` = numero totale di pixel appartenenti alla classe.
- `pixel_percentage` = percentuale sui **pixel validi** (etichette in range); somma a 100.
- `ignored_pixels` = pixel con etichetta fuori range, esclusi dalle percentuali.

In [ ]:
# ============================================================
# STATISTICHE COMPLETE: PER SPLIT, PER VIDEO, GLOBALI
# ============================================================

# Chiave di raggruppamento (se assente nei sample si usa "surgery_id", es. DSAD)
VIDEO_KEY = "video"

all_indices = sorted({i for idx in splits.values() for i in idx})

# Ogni maschera viene letta una sola volta
sample_cache = compute_sample_counts(dataset, all_indices)

# --- per split ---
statistics = pd.concat(
    [
        class_pixel_statistics(dataset, indices, split_name, cache=sample_cache)
        for split_name, indices in splits.items()
    ],
    ignore_index=True,
)

# --- per video (dentro ogni split) ---
video_statistics = pd.concat(
    [
        class_statistics_by_video(
            dataset, indices, group_key=VIDEO_KEY,
            split_name=split_name, cache=sample_cache,
        )
        for split_name, indices in splits.items()
        if len(indices)
    ],
    ignore_index=True,
)

# --- globale ---
global_statistics = class_statistics_global(dataset, all_indices, cache=sample_cache)

# --- in quali video compare ogni classe ---
video_presence = pd.concat(
    [
        class_video_presence(
            dataset, indices, group_key=VIDEO_KEY, cache=sample_cache, split=split_name
        )
        for split_name, indices in splits.items()
        if len(indices)
    ],
    ignore_index=True,
)
global_video_presence = class_video_presence(
    dataset, all_indices, group_key=VIDEO_KEY, cache=sample_cache, scope="global"
)
# frame che contengono ogni classe: righe = video, colonne = classi
global_matrix = class_video_matrix(
    dataset, all_indices, group_key=VIDEO_KEY, cache=sample_cache
)

display(statistics)


In [ ]:
# ============================================================
# TABELLA COMPATTA PER IL REPORT / TESI
# ============================================================

report_table = statistics[
    [
        "split",
        "class",
        "samples",
        "sample_percentage",
        "pixels",
        "pixel_percentage",
    ]
].copy()

report_table["sample_percentage"] = report_table[
    "sample_percentage"
].round(2)

report_table["pixel_percentage"] = report_table[
    "pixel_percentage"
].round(2)

display(report_table)


## Controllo rapido

La somma delle `pixel_percentage` di ogni split / video / globale deve essere **100%**. I pixel con etichetta fuori range sono riportati in `ignored_pixels` (atteso: 0).

In [ ]:
# ============================================================
# SANITY CHECK PIXEL DISTRIBUTION
# ============================================================

def sanity(df, keys):
    out = (
        df.groupby(keys, as_index=False)
        .agg(
            pixel_percentage_sum=("pixel_percentage", "sum"),
            ignored_pixels=("ignored_pixels", "first"),
            valid_pixels=("valid_pixels", "first"),
        )
    )
    out["pixel_percentage_sum"] = out["pixel_percentage_sum"].round(4)
    return out


print("Per split")
display(sanity(statistics, ["split"]))

print("Per video")
display(sanity(video_statistics, ["split", "video"]))

print("Globale")
display(sanity(global_statistics, ["scope"]))

# Coerenza: somma dei pixel per split == pixel globali (se gli split coprono tutto il dataset)
check = (
    statistics.groupby("class_id")["pixels"].sum()
    .eq(global_statistics.set_index("class_id")["pixels"])
    .all()
)
print(f"Somma pixel degli split == globale: {check}")


## Presenza delle classi nei video (class imbalance)

Per ogni classe: **in quanti video compare** e **quali**. Una classe presente in molti frame ma in pochi video ha una dimensione campionaria effettiva molto bassa (i frame dello stesso video sono correlati).

- `videos` / `total_videos` / `video_percentage`: numero e percentuale di video che contengono la classe;
- `video_list`: i video che la contengono;
- `frames_per_video`: quanti frame di ciascun video la contengono;
- `top_video`, `top_video_pixel_share`: video con più pixel della classe e quota % dei pixel della classe in quel video (100 = la classe vive in un solo video).

In [ ]:
# ============================================================
# VIDEO CHE CONTENGONO OGNI CLASSE
# ============================================================

RARE_VIDEO_THRESHOLD = 3        # classe in <= N video
CONCENTRATION_THRESHOLD = 50.0  # >= X% dei pixel della classe in un solo video

print("GLOBALE")
display(
    global_video_presence[
        ["class_id", "class", "videos", "total_videos", "video_percentage",
         "top_video", "top_video_pixel_share", "video_list"]
    ].round(1)
)

print("PER SPLIT")
display(
    video_presence[
        ["split", "class_id", "class", "videos", "total_videos", "video_percentage",
         "top_video", "top_video_pixel_share", "video_list"]
    ].round(1)
)

# Avvisi di class imbalance
warnings_df = global_video_presence[
    (global_video_presence["videos"] <= RARE_VIDEO_THRESHOLD)
    | (
        (global_video_presence["top_video_pixel_share"] >= CONCENTRATION_THRESHOLD)
        & (global_video_presence["videos"] > 0)
    )
]
print(f"Classi presenti in <= {RARE_VIDEO_THRESHOLD} video o concentrate "
      f"(>= {CONCENTRATION_THRESHOLD:.0f}% pixel in un video):")
display(warnings_df[["class", "videos", "total_videos", "top_video",
                     "top_video_pixel_share", "video_list"]].round(1))

# Classi assenti da uno split
missing = video_presence[video_presence["videos"] == 0][["split", "class"]]
print("Classi assenti da uno split:")
display(missing if len(missing) else "nessuna")

# Dettaglio: frame per video che contengono la classe
print("Frame contenenti la classe, per video")
display(global_matrix)


In [ ]:
# ============================================================
# GRAFICO: VIDEO PER CLASSE + HEATMAP VIDEO x CLASSE
# ============================================================

import matplotlib.pyplot as plt

colors = [
    "#d9534f" if (v <= RARE_VIDEO_THRESHOLD or s >= CONCENTRATION_THRESHOLD) else "#4c72b0"
    for v, s in zip(global_video_presence["videos"],
                    global_video_presence["top_video_pixel_share"])
]
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(global_video_presence["class"], global_video_presence["videos"], color=colors)
ax.axhline(global_video_presence["total_videos"].iloc[0], color="gray", ls="--", lw=0.8)
ax.set_ylabel("Video che contengono la classe")
ax.set_title("Video per classe (rosso = possibile class imbalance)")
ax.tick_params(axis="x", rotation=45)
for lbl in ax.get_xticklabels():
    lbl.set_ha("right")
plt.tight_layout()
plt.show()

_samples = get_samples(dataset)
_key = VIDEO_KEY if VIDEO_KEY in _samples[0] else "surgery_id"
video_frames = pd.Series(
    {v: sum(1 for i in all_indices if str(_samples[i][_key]) == v)
     for v in global_matrix.index}
)
heat_pct = global_matrix.div(video_frames, axis=0) * 100

fig, ax = plt.subplots(figsize=(14, max(4, 0.35 * len(heat_pct))))
im = ax.imshow(heat_pct.values, aspect="auto", cmap="viridis", vmin=0, vmax=100)
ax.set_xticks(range(heat_pct.shape[1]))
ax.set_xticklabels(heat_pct.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heat_pct)))
ax.set_yticklabels(heat_pct.index)
ax.set_title("% di frame del video che contengono la classe")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## Statistiche per video

Stesse metriche calcolate separatamente per ogni video (dentro il rispettivo split). Utile per capire quali video contengono le classi rare.

In [ ]:
# ============================================================
# STATISTICHE PER VIDEO
# ============================================================

display(
    video_statistics[
        ["split", "video", "class", "samples", "sample_percentage",
         "pixels", "pixel_percentage", "total_samples"]
    ].round(2)
)

# Matrici video x classe
pixel_by_video = video_statistics.pivot_table(
    index=["split", "video"], columns="class", values="pixel_percentage", sort=False
).round(2)
presence_by_video = video_statistics.pivot_table(
    index=["split", "video"], columns="class", values="sample_percentage", sort=False
).round(2)

print("Pixel (%) per video")
display(pixel_by_video)

print("Presenza (% frame) per video")
display(presence_by_video)


In [ ]:
# ============================================================
# HEATMAP VIDEO x CLASSE
# ============================================================

import matplotlib.pyplot as plt

for title, table, cmap in [
    ("Presenza della classe (% frame) per video", presence_by_video, "viridis"),
    ("Pixel (%) per video", pixel_by_video, "magma"),
]:
    fig, ax = plt.subplots(figsize=(14, max(3, 0.4 * len(table))))
    im = ax.imshow(table.values, aspect="auto", cmap=cmap)
    ax.set_xticks(range(len(table.columns)))
    ax.set_xticklabels(table.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(table.index)))
    ax.set_yticklabels([f"{s} / {v}" for s, v in table.index])
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


## Statistiche globali

Stesse metriche sull'intero dataset (tutti i campioni, indipendentemente dallo split).

In [ ]:
# ============================================================
# STATISTICHE GLOBALI
# ============================================================

display(global_statistics.round(4))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].bar(global_statistics["class"], global_statistics["pixel_percentage"])
axes[0].set_ylabel("Pixel percentage (%)")
axes[0].set_title("Pixel distribution - global")
axes[1].bar(global_statistics["class"], global_statistics["sample_percentage"])
axes[1].set_ylabel("Samples containing class (%)")
axes[1].set_ylim(0, 100)
axes[1].set_title("Class presence - global")
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# PIXEL DISTRIBUTION
# ============================================================

import matplotlib.pyplot as plt


for split_name in statistics["split"].unique():

    split_stats = statistics[
        statistics["split"] == split_name
    ].copy()

    plt.figure(figsize=(12, 6))

    plt.bar(
        split_stats["class"],
        split_stats["pixel_percentage"],
    )

    plt.xlabel("Class")
    plt.ylabel("Pixel percentage (%)")
    plt.title(
        f"Pixel distribution - {split_name}"
    )

    plt.xticks(
        rotation=45,
        ha="right",
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# PIXEL DISTRIBUTION - TRAIN VS VALIDATION VS TEST
# ============================================================

import matplotlib.pyplot as plt
import numpy as np


pivot = statistics.pivot(
    index="class",
    columns="split",
    values="pixel_percentage",
).fillna(0)

x = np.arange(len(pivot.index))
splits_to_plot = list(pivot.columns)

width = 0.8 / len(splits_to_plot)

plt.figure(figsize=(14, 6))

for i, split_name in enumerate(splits_to_plot):

    values = pivot[split_name].values

    plt.bar(
        x + i * width,
        values,
        width,
        label=split_name,
    )

plt.xlabel("Class")
plt.ylabel("Pixel percentage (%)")
plt.title("Pixel distribution across dataset splits")

plt.xticks(
    x + width * (len(splits_to_plot) - 1) / 2,
    pivot.index,
    rotation=45,
    ha="right",
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CLASS PRESENCE IN SAMPLES
# ============================================================

import matplotlib.pyplot as plt


for split_name in statistics["split"].unique():

    split_stats = statistics[
        statistics["split"] == split_name
    ].copy()

    plt.figure(figsize=(12, 6))

    plt.bar(
        split_stats["class"],
        split_stats["sample_percentage"],
    )

    plt.xlabel("Class")
    plt.ylabel("Samples containing class (%)")
    plt.title(
        f"Class presence - {split_name}"
    )

    plt.xticks(
        rotation=45,
        ha="right",
    )

    plt.ylim(0, 100)

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# QUALITATIVE VISUALIZATION
# ============================================================

import matplotlib.pyplot as plt
import numpy as np


def prepare_image(image):
    """
    Convert image to HWC format suitable for matplotlib.
    """

    image = np.asarray(image)

    if image.ndim == 2:
        return image

    if image.ndim == 3:

        # CHW -> HWC
        if image.shape[0] in (1, 3, 4):
            image = np.transpose(
                image,
                (1, 2, 0),
            )

        # Remove single channel dimension
        if image.shape[-1] == 1:
            image = image[..., 0]

    return image


def normalize_image(image):
    """
    Normalize image for visualization.
    """

    image = image.astype(np.float32)

    min_value = image.min()
    max_value = image.max()

    if max_value > min_value:
        image = (
            image - min_value
        ) / (
            max_value - min_value
        )

    return image


def visualize_sample(
    dataset,
    index,
    class_names,
    alpha=0.5,
):
    """
    Display:
        1. Original image
        2. Ground truth
        3. Overlay
    """

    sample = load_raw_sample(
        dataset,
        index,
    )

    image = prepare_image(
        to_numpy(sample["image"])
    )

    mask = to_numpy(
        sample["mask"]
    )

    image = normalize_image(image)

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(18, 6),
    )

    # --------------------------------------------------------
    # Original image
    # --------------------------------------------------------

    axes[0].imshow(image)

    axes[0].set_title(
        f"Image - index {index}"
    )

    axes[0].axis("off")

    # --------------------------------------------------------
    # Ground truth
    # --------------------------------------------------------

    axes[1].imshow(
        mask,
        interpolation="nearest",
    )

    axes[1].set_title(
        "Ground Truth"
    )

    axes[1].axis("off")

    # --------------------------------------------------------
    # Overlay
    # --------------------------------------------------------

    axes[2].imshow(image)

    axes[2].imshow(
        mask,
        alpha=alpha,
        interpolation="nearest",
    )

    axes[2].set_title(
        "Ground Truth Overlay"
    )

    axes[2].axis("off")

    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------

    present_classes = np.unique(mask)

    legend_text = []

    for class_id in present_classes:

        class_id = int(class_id)

        if 0 <= class_id < len(class_names):
            legend_text.append(
                f"{class_id}: {class_names[class_id]}"
            )

    fig.text(
        0.5,
        0.01,
        " | ".join(legend_text),
        ha="center",
        fontsize=10,
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# RANDOM SAMPLES
# ============================================================

import random


def visualize_random_samples(
    dataset,
    indices,
    class_names,
    n_samples=5,
):

    indices = list(indices)

    n_samples = min(
        n_samples,
        len(indices),
    )

    selected_indices = random.sample(
        indices,
        n_samples,
    )

    for index in selected_indices:

        visualize_sample(
            dataset=dataset,
            index=index,
            class_names=class_names,
        )

In [ ]:
class_names = get_class_names(dataset)

visualize_random_samples(
    dataset=dataset,
    indices=splits["validation"],
    class_names=class_names,
    n_samples=5,
)

In [ ]:
# ============================================================
# GENERATE DATASET VALIDATION REPORT
# ============================================================
#
# Struttura del PDF:
#   Cover
#   1. Executive summary          (punti chiave + avvisi di class imbalance)
#   2. Dataset overview           (split, video, classi)
#   3. Global statistics          (tutto il dataset)
#   4. Per-split statistics       (una sezione per split)
#   5. Video coverage             (in quali video compare ogni classe)
#   6. Cross-split comparison
#   7. Qualitative analysis       (3 campioni per pagina)
#   8. Conclusions
#
# Vengono inoltre salvati i CSV di tutte le tabelle.

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from matplotlib.backends.backend_pdf import PdfPages


# ============================================================
# CONFIGURATION
# ============================================================

REPORT_DIR = PROJECT_ROOT / "docs" / "reports" / "dataset_validation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = DATASET_CONFIG.split("/")[-1]
PDF_PATH = REPORT_DIR / f"{DATASET_NAME}_validation_report.pdf"

N_QUALITATIVE_SAMPLES = 6      # per split
SAMPLES_PER_PAGE = 3
ROWS_PER_TABLE_PAGE = 22

# Soglie per gli avvisi di class imbalance
RARE_VIDEO_THRESHOLD = 3       # classe presente in <= N video
CONCENTRATION_THRESHOLD = 50.0 # >= X% dei pixel della classe in un solo video

PAGE_W, PAGE_H = 8.27, 11.69   # A4 portrait
FIGURE_SIZE = (PAGE_W, PAGE_H)
LANDSCAPE = (PAGE_H, PAGE_W)

WARN_COLOR = "#fde2e2"
HEADER_COLOR = "#e8eaf0"


# ============================================================
# DATA
# ============================================================

statistics_df = statistics.copy()
class_names = get_class_names(dataset)
split_names = [s for s in splits if len(splits[s]) > 0]

all_samples = get_samples(dataset)
VIDEO_COL = VIDEO_KEY if all_samples and VIDEO_KEY in all_samples[0] else "surgery_id"


def n_videos(indices):
    return len({str(all_samples[i][VIDEO_COL]) for i in indices})


video_presence = pd.concat(
    [
        class_video_presence(
            dataset, splits[s], group_key=VIDEO_KEY, cache=sample_cache, split=s
        )
        for s in split_names
    ],
    ignore_index=True,
)
global_video_presence = class_video_presence(
    dataset, all_indices, group_key=VIDEO_KEY, cache=sample_cache, scope="global"
)
global_matrix = class_video_matrix(
    dataset, all_indices, group_key=VIDEO_KEY, cache=sample_cache
)

# Tabella globale completa (frame + pixel + video)
global_full = global_statistics.merge(
    global_video_presence[
        ["class_id", "videos", "total_videos", "video_percentage",
         "video_list", "top_video", "top_video_pixel_share"]
    ],
    on="class_id",
)

split_full = statistics_df.merge(
    video_presence[
        ["split", "class_id", "videos", "total_videos", "video_percentage",
         "video_list", "top_video", "top_video_pixel_share"]
    ],
    on=["split", "class_id"],
)

# Avvisi di class imbalance (livello globale)
global_full["flag_rare"] = global_full["videos"] <= RARE_VIDEO_THRESHOLD
global_full["flag_concentrated"] = (
    global_full["top_video_pixel_share"] >= CONCENTRATION_THRESHOLD
) & (global_full["videos"] > 0)
global_full["flag_missing"] = global_full["videos"] == 0
global_full["flag_any"] = (
    global_full["flag_rare"] | global_full["flag_concentrated"]
)

# Classi assenti / presenti in pochi video per split
split_missing = split_full[split_full["videos"] == 0]


# ============================================================
# CSV EXPORT
# ============================================================

def _csv_ready(df):
    df = df.copy()
    for col in df.columns:
        if df[col].map(lambda x: isinstance(x, (list, dict))).any():
            df[col] = df[col].map(
                lambda x: ";".join(map(str, x)) if isinstance(x, list)
                else ";".join(f"{k}:{v}" for k, v in x.items())
                if isinstance(x, dict) else x
            )
    return df


for name, df in {
    "split_statistics": split_full,
    "video_statistics": video_statistics,
    "global_statistics": global_full,
    "class_video_matrix": global_matrix.reset_index(names="video"),
}.items():
    _csv_ready(df).to_csv(
        REPORT_DIR / f"{DATASET_NAME}_{name}.csv", index=False
    )


# ============================================================
# HELPERS
# ============================================================

page_counter = {"n": 0}


def save_page(pdf, fig, footer=True, **savefig_kwargs):
    """Save a page with a consistent footer and an automatic page number."""
    page_counter["n"] += 1
    if footer:
        w, h = fig.get_size_inches()
        fig.text(0.07, 0.02, f"Dataset Validation Report - {DATASET_NAME}",
                 fontsize=7, color="gray")
        fig.text(0.93, 0.02, f"Page {page_counter['n']}", fontsize=7,
                 color="gray", ha="right")
    pdf.savefig(fig, **savefig_kwargs)
    plt.close(fig)


def new_page(title, subtitle=None, size=FIGURE_SIZE):
    fig = plt.figure(figsize=size)
    fig.text(0.07, 0.95, title, fontsize=18, fontweight="bold", va="top")
    if subtitle:
        fig.text(0.07, 0.915, subtitle, fontsize=9.5, va="top", color="#444")
    return fig


def draw_table(ax, df, columns, headers, font_size=7.5, scale=(1, 1.45),
               highlight=None, col_widths=None):
    ax.axis("off")
    table = ax.table(
        cellText=df[columns].values.tolist(),
        colLabels=headers,
        loc="upper center",
        cellLoc="center",
        colWidths=col_widths,
    )
    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.scale(*scale)
    for (row, col), cell in table.get_celld().items():
        cell.set_linewidth(0.4)
        if row == 0:
            cell.set_facecolor(HEADER_COLOR)
            cell.set_text_props(fontweight="bold")
        elif highlight is not None and bool(highlight.iloc[row - 1]):
            cell.set_facecolor(WARN_COLOR)
        if col == 0 and row > 0:
            cell.set_text_props(ha="left")
    return table


def paginate(df, size):
    for start in range(0, len(df), size):
        yield df.iloc[start:start + size]


def shorten_videos(videos, limit=6):
    videos = list(videos)
    if len(videos) <= limit:
        return ", ".join(videos) if videos else "-"
    return ", ".join(videos[:limit]) + f", ... (+{len(videos) - limit})"


def barh_chart(ax, df, value_col, xlabel, title, xlim=None):
    ordered = df.sort_values(value_col)
    ax.barh(ordered["class"], ordered[value_col])
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontsize=10)
    ax.grid(axis="x", alpha=0.3)
    ax.tick_params(axis="y", labelsize=7)
    if xlim:
        ax.set_xlim(*xlim)


def prepare_image_for_plot(image):
    image = np.asarray(image)
    if image.ndim == 3:
        if image.shape[0] in (1, 3, 4) and image.shape[-1] not in (1, 3, 4):
            image = np.moveaxis(image, 0, -1)
        if image.shape[-1] == 1:
            image = image[..., 0]
    image = image.astype(np.float32)
    lo, hi = image.min(), image.max()
    return (image - lo) / (hi - lo) if hi > lo else image


def get_image_and_mask(dataset, index):
    sample = load_raw_sample(dataset, int(index))
    if isinstance(sample, dict):
        image = sample.get("image", sample.get("img"))
        mask = sample.get("mask", sample.get("label"))
        if image is None or mask is None:
            raise ValueError(f"Unable to find image/mask. Keys: {list(sample)}")
        return to_numpy(image), to_numpy(mask)
    return to_numpy(sample[0]), to_numpy(sample[1])


def fmt_int(x):
    return f"{int(x):,}"


# ============================================================
# REPORT
# ============================================================

total_videos_global = global_full["total_videos"].iloc[0]
ignored_global = int(global_statistics["ignored_pixels"].iloc[0])

with PdfPages(PDF_PATH) as pdf:

    # ========================================================
    # COVER
    # ========================================================
    fig = plt.figure(figsize=FIGURE_SIZE)
    fig.text(0.08, 0.72, "Dataset Validation", fontsize=30, fontweight="bold")
    fig.text(0.08, 0.65, "Surgical Scene Semantic Segmentation", fontsize=17)
    fig.text(0.08, 0.56, f"Dataset: {DATASET_NAME}", fontsize=15, fontweight="bold")
    fig.text(0.08, 0.51, f"Classes: {len(class_names)}   |   "
                         f"Videos: {total_videos_global}   |   "
                         f"Frames: {len(all_indices):,}", fontsize=11)
    fig.text(0.08, 0.15,
             "Class presence is analysed at three levels - frames, pixels and "
             "videos - on the complete dataset (no sampling), globally and for "
             "every split. Video coverage highlights class imbalance that "
             "frame and pixel counts alone can hide.",
             fontsize=9.5, wrap=True, va="top")
    save_page(pdf, fig, footer=False)

    # ========================================================
    # 1. EXECUTIVE SUMMARY
    # ========================================================
    fig = new_page("1. Executive Summary", f"Dataset: {DATASET_NAME}")

    lines = [
        ("Dataset", None),
        (f"{len(class_names)} classes, {total_videos_global} videos, "
         f"{len(all_indices):,} frames.", "b"),
    ]
    for s in split_names:
        lines.append(
            (f"{s}: {len(splits[s]):,} frames, {n_videos(splits[s])} videos", "b")
        )
    lines.append((f"Ignored (out-of-range) pixels: {ignored_global:,}"
                  + ("" if ignored_global == 0 else "  <-- check labels!"), "b"))

    lines.append(("", None))
    lines.append((f"Class imbalance warnings (global, thresholds: "
                  f"<= {RARE_VIDEO_THRESHOLD} videos, "
                  f">= {CONCENTRATION_THRESHOLD:.0f}% pixels in one video)", None))

    flagged = global_full[global_full["flag_any"]].sort_values("videos")
    if flagged.empty:
        lines.append(("No class is rare or concentrated in few videos.", "b"))
    for _, r in flagged.iterrows():
        lines.append((
            f"{r['class']}: in {r['videos']}/{r['total_videos']} videos "
            f"({r['video_percentage']:.0f}%), {r['top_video_pixel_share']:.0f}% "
            f"of its pixels in {r['top_video']}", "b"))

    lines.append(("", None))
    lines.append(("Classes missing from a split", None))
    if split_missing.empty:
        lines.append(("Every class appears in every split.", "b"))
    else:
        for s in split_names:
            miss = split_missing[split_missing["split"] == s]["class"].tolist()
            if miss:
                lines.append((f"{s}: {', '.join(miss)}", "b"))

    lines.append(("", None))
    lines.append(("Dominant / smallest pixel share (global)", None))
    dom = global_full.loc[global_full["pixel_percentage"].idxmax()]
    nz = global_full[global_full["pixels"] > 0]
    small = nz.loc[nz["pixel_percentage"].idxmin()]
    lines.append((f"Dominant: {dom['class']} ({dom['pixel_percentage']:.2f}%)", "b"))
    lines.append((f"Smallest: {small['class']} ({small['pixel_percentage']:.4f}%)", "b"))

    y = 0.87
    for text, kind in lines:
        if kind is None and text:
            y -= 0.012
            fig.text(0.08, y, text, fontsize=11, fontweight="bold", va="top")
            y -= 0.027
        elif kind == "b":
            fig.text(0.10, y, "- " + text, fontsize=8.8, va="top", wrap=True)
            y -= 0.022
        else:
            y -= 0.012
    save_page(pdf, fig)

    # ========================================================
    # 2. DATASET OVERVIEW
    # ========================================================
    fig = new_page("2. Dataset Overview", f"Dataset: {DATASET_NAME}")
    overview_df = pd.DataFrame([
        {"Split": s, "Frames": fmt_int(len(splits[s])), "Videos": n_videos(splits[s])}
        for s in split_names
    ] + [{"Split": "global", "Frames": fmt_int(len(all_indices)),
          "Videos": total_videos_global}])
    ax = fig.add_axes([0.08, 0.70, 0.84, 0.17])
    draw_table(ax, overview_df, ["Split", "Frames", "Videos"],
               ["Split", "Frames", "Videos"], font_size=10, scale=(1, 1.8))

    fig.text(0.08, 0.64, "Classes", fontsize=13, fontweight="bold")
    col_size = 24
    for k in range(0, len(class_names), col_size):
        chunk = class_names[k:k + col_size]
        fig.text(0.08 + 0.45 * (k // col_size), 0.61,
                 "\n".join(f"{k + j:>2}  {n}" for j, n in enumerate(chunk)),
                 fontsize=8.5, va="top", family="monospace", linespacing=1.5)
    save_page(pdf, fig)

    # ========================================================
    # 3. GLOBAL STATISTICS
    # ========================================================
    g = global_full.copy()
    g["Frames"] = g["samples"].map(fmt_int)
    g["Frames %"] = g["sample_percentage"].map(lambda x: f"{x:.2f}")
    g["Pixels"] = g["pixels"].map(fmt_int)
    g["Pixels %"] = g["pixel_percentage"].map(lambda x: f"{x:.4f}")
    g["Videos"] = g.apply(lambda r: f"{r['videos']}/{r['total_videos']}", axis=1)

    for part, chunk in enumerate(paginate(g, ROWS_PER_TABLE_PAGE), start=1):
        fig = new_page("3. Global Statistics",
                       f"All splits together - rows highlighted = class imbalance warning"
                       f"  (part {part})")
        ax = fig.add_axes([0.07, 0.30, 0.86, 0.60])
        draw_table(ax, chunk,
                   ["class", "Frames", "Frames %", "Pixels", "Pixels %", "Videos"],
                   ["Class", "Frames", "Frames %", "Pixels", "Pixels %", "Videos"],
                   highlight=chunk["flag_any"],
                   col_widths=[0.30, 0.12, 0.12, 0.18, 0.12, 0.12])
        if part == 1:
            cax = fig.add_axes([0.25, 0.06, 0.65, 0.18])
            barh_chart(cax, global_full, "pixel_percentage", "Pixels (%)",
                       "Global pixel distribution")
        save_page(pdf, fig)

    # ========================================================
    # 4. PER-SPLIT STATISTICS
    # ========================================================
    for s in split_names:
        d = split_full[split_full["split"] == s].sort_values("class_id").copy()
        d["Frames"] = d["samples"].map(fmt_int)
        d["Frames %"] = d["sample_percentage"].map(lambda x: f"{x:.2f}")
        d["Pixels"] = d["pixels"].map(fmt_int)
        d["Pixels %"] = d["pixel_percentage"].map(lambda x: f"{x:.4f}")
        d["Videos"] = d.apply(lambda r: f"{r['videos']}/{r['total_videos']}", axis=1)
        d["warn"] = (d["videos"] <= RARE_VIDEO_THRESHOLD)

        for part, chunk in enumerate(paginate(d, ROWS_PER_TABLE_PAGE), start=1):
            fig = new_page(f"4. Split Statistics - {s}",
                           f"{len(splits[s]):,} frames, {n_videos(splits[s])} videos "
                           f"- highlighted = class in <= {RARE_VIDEO_THRESHOLD} videos"
                           f"  (part {part})")
            ax = fig.add_axes([0.07, 0.30, 0.86, 0.60])
            draw_table(ax, chunk,
                       ["class", "Frames", "Frames %", "Pixels", "Pixels %", "Videos"],
                       ["Class", "Frames", "Frames %", "Pixels", "Pixels %", "Videos"],
                       highlight=chunk["warn"],
                       col_widths=[0.30, 0.12, 0.12, 0.18, 0.12, 0.12])
            if part == 1:
                cax = fig.add_axes([0.25, 0.06, 0.65, 0.18])
                barh_chart(cax, d, "pixel_percentage", "Pixels (%)",
                           f"Pixel distribution - {s}")
            save_page(pdf, fig)

    # ========================================================
    # 5. VIDEO COVERAGE
    # ========================================================
    # 5a. Global table: how many videos and which ones
    gv = global_full.copy()
    gv["Videos"] = gv.apply(lambda r: f"{r['videos']}/{r['total_videos']}", axis=1)
    gv["Videos %"] = gv["video_percentage"].map(lambda x: f"{x:.0f}")
    gv["Top video"] = gv["top_video"].fillna("-")
    gv["Top share %"] = gv["top_video_pixel_share"].map(lambda x: f"{x:.0f}")
    gv["Which videos"] = gv["video_list"].map(lambda v: shorten_videos(v, 5))

    for part, chunk in enumerate(paginate(gv, ROWS_PER_TABLE_PAGE), start=1):
        fig = new_page(
            "5. Video Coverage",
            "In how many (and which) videos each class appears - global. "
            "Top share = % of the class pixels in its dominant video "
            f"(highlighted: <= {RARE_VIDEO_THRESHOLD} videos or >= "
            f"{CONCENTRATION_THRESHOLD:.0f}%)  (part {part})")
        ax = fig.add_axes([0.04, 0.12, 0.92, 0.76])
        draw_table(ax, chunk,
                   ["class", "Videos", "Videos %", "Top video", "Top share %",
                    "Which videos"],
                   ["Class", "Videos", "Videos %", "Top video", "Top share %",
                    "Which videos"],
                   font_size=7, highlight=chunk["flag_any"],
                   col_widths=[0.20, 0.08, 0.08, 0.10, 0.09, 0.45])
        save_page(pdf, fig)

    # 5b. Per-split video coverage
    for s in split_names:
        d = split_full[split_full["split"] == s].sort_values("class_id").copy()
        d["Videos"] = d.apply(lambda r: f"{r['videos']}/{r['total_videos']}", axis=1)
        d["Top video"] = d["top_video"].fillna("-")
        d["Top share %"] = d["top_video_pixel_share"].map(lambda x: f"{x:.0f}")
        d["Which videos"] = d["video_list"].map(lambda v: shorten_videos(v, 5))
        d["warn"] = d["videos"] <= RARE_VIDEO_THRESHOLD

        for part, chunk in enumerate(paginate(d, ROWS_PER_TABLE_PAGE), start=1):
            fig = new_page(f"5. Video Coverage - {s}",
                           f"{n_videos(splits[s])} videos in this split  (part {part})")
            ax = fig.add_axes([0.04, 0.12, 0.92, 0.76])
            draw_table(ax, chunk,
                       ["class", "Videos", "Top video", "Top share %", "Which videos"],
                       ["Class", "Videos", "Top video", "Top share %", "Which videos"],
                       font_size=7, highlight=chunk["warn"],
                       col_widths=[0.22, 0.09, 0.11, 0.10, 0.48])
            save_page(pdf, fig)

    # 5c. Heatmaps video x class (frames containing the class)
    heat = global_matrix.copy()
    video_split = {
        str(all_samples[i][VIDEO_COL]): s
        for s in split_names for i in splits[s]
    }
    heat = heat.loc[sorted(heat.index, key=lambda v: (video_split.get(v, ""), v))]
    heat_pct = heat.div(
        pd.Series({v: sum(1 for i in all_indices
                          if str(all_samples[i][VIDEO_COL]) == v)
                   for v in heat.index}),
        axis=0,
    ) * 100

    fig, ax = plt.subplots(figsize=(11.69, max(5, 0.3 * len(heat) + 2.5)))
    im = ax.imshow(heat_pct.values, aspect="auto", cmap="viridis", vmin=0, vmax=100)
    ax.set_xticks(range(heat.shape[1]))
    ax.set_xticklabels(heat.columns, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(len(heat)))
    ax.set_yticklabels([f"{v} [{video_split.get(v, '?')}]" for v in heat.index],
                       fontsize=7)
    ax.set_title("Class presence per video (% of the video's frames containing "
                 "the class)", fontsize=12, fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.025)
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    save_page(pdf, fig)

    # 5d. Videos per class (bar)
    fig, ax = plt.subplots(figsize=(11.69, 6))
    colors = [
        "#d9534f" if f else "#4c72b0" for f in global_full["flag_any"]
    ]
    ax.bar(global_full["class"], global_full["videos"], color=colors)
    ax.axhline(total_videos_global, color="gray", ls="--", lw=0.8)
    ax.set_ylabel("Number of videos containing the class")
    ax.set_title("Videos per class (red = imbalance warning, dashed = total videos)",
                 fontsize=12, fontweight="bold")
    ax.tick_params(axis="x", rotation=60, labelsize=8)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    save_page(pdf, fig)

    # ========================================================
    # 6. CROSS-SPLIT COMPARISON
    # ========================================================
    for title, col, ylabel, ylim in [
        ("Class presence across splits", "sample_percentage",
         "Frames containing class (%)", (0, 100)),
        ("Pixel distribution across splits", "pixel_percentage", "Pixels (%)", None),
        ("Video coverage across splits", "video_percentage",
         "Videos containing class (%)", (0, 100)),
    ]:
        pivot = split_full.pivot(index="class", columns="split", values=col).fillna(0)
        pivot = pivot.loc[class_names]
        fig, ax = plt.subplots(figsize=(11.69, 6))
        pivot[split_names].plot(kind="bar", ax=ax, width=0.8)
        ax.set_title(f"6. {title}", fontsize=13, fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel(ylabel)
        if ylim:
            ax.set_ylim(*ylim)
        ax.grid(axis="y", alpha=0.3)
        ax.legend(title="Split")
        plt.setp(ax.get_xticklabels(), rotation=60, ha="right", fontsize=8)
        fig.tight_layout(rect=(0, 0.03, 1, 1))
        save_page(pdf, fig)

    # ========================================================
    # 7. QUALITATIVE ANALYSIS (3 samples per page)
    # ========================================================
    for s in split_names:
        rng = np.random.default_rng(seed=42)
        chosen = rng.choice(
            splits[s], size=min(N_QUALITATIVE_SAMPLES, len(splits[s])), replace=False
        )
        for page_start in range(0, len(chosen), SAMPLES_PER_PAGE):
            batch_idx = chosen[page_start:page_start + SAMPLES_PER_PAGE]
            fig, axes = plt.subplots(len(batch_idx), 3,
                                     figsize=FIGURE_SIZE, squeeze=False)
            fig.suptitle(f"7. Qualitative Analysis - {s}", fontsize=15,
                         fontweight="bold", y=0.985)
            for row, index in enumerate(batch_idx):
                image, mask = get_image_and_mask(dataset, int(index))
                image = prepare_image_for_plot(image)
                mask = np.squeeze(mask)
                video = all_samples[int(index)][VIDEO_COL]
                present = [class_names[c] for c in np.unique(mask)
                           if 0 <= int(c) < len(class_names)]

                axes[row, 0].imshow(image)
                axes[row, 0].set_title(f"Image - #{int(index)} ({video})", fontsize=8)
                axes[row, 1].imshow(mask, interpolation="nearest",
                                    vmin=0, vmax=len(class_names) - 1)
                axes[row, 1].set_title("Ground truth", fontsize=8)
                axes[row, 2].imshow(image)
                axes[row, 2].imshow(mask, alpha=0.5, interpolation="nearest",
                                    vmin=0, vmax=len(class_names) - 1)
                axes[row, 2].set_title("Overlay", fontsize=8)
                for a in axes[row]:
                    a.axis("off")
                axes[row, 0].text(
                    0, -0.08, "Classes: " + ", ".join(present), fontsize=6.5,
                    transform=axes[row, 0].transAxes, va="top", wrap=True)
            fig.tight_layout(rect=(0, 0.04, 1, 0.96))
            save_page(pdf, fig)

    # ========================================================
    # 8. CONCLUSIONS
    # ========================================================
    fig = new_page("8. Conclusions", f"Dataset: {DATASET_NAME}")
    notes = [
        "Frames, pixels and videos measure three different things:",
        "  - Frames: how often the class occurs across images.",
        "  - Pixels: how much image area the class occupies.",
        "  - Videos: how many independent patients/procedures show the class.",
        "",
        "A class present in many frames but in only a few videos is a weak "
        "basis for generalisation: frames of the same video are highly "
        "correlated, so the effective sample size is closer to the number of "
        "videos than to the number of frames.",
        "",
        "Splits are made at video level, so a rare class can end up entirely "
        "in train or entirely in validation/test. Check 'Classes missing from a "
        "split' in the executive summary before interpreting per-class metrics.",
        "",
        "Classes flagged in red (few videos, or concentrated in one video) "
        "deserve special care: per-class metrics on them are high-variance, "
        "and class weighting or oversampling should be evaluated.",
    ]
    fig.text(0.08, 0.88, "\n".join(notes), fontsize=9.5, va="top",
             wrap=True, linespacing=1.6)
    save_page(pdf, fig)


# ============================================================
# FINAL MESSAGE
# ============================================================

print("=" * 70)
print("DATASET VALIDATION REPORT")
print("=" * 70)
print(f"Dataset : {DATASET_NAME}")
print(f"Pages   : {page_counter['n']}")
print(f"PDF     : {PDF_PATH.resolve()}")
print(f"CSV     : {REPORT_DIR.resolve()}/{DATASET_NAME}_*.csv")
